# Árbol de decisión (versión rápida)
Notebook autónomo para probar el árbol de decisión en otra PC. Reproduce la limpieza y el preprocesamiento de `primer_parcial.ipynb`, con dos cambios para que corra más rápido:

1. **Los pasos que no aprenden de los datos se aplican una sola vez.** Antes se repetían en cada uno de los cientos de entrenamientos de la búsqueda de hiperparámetros: eliminar columnas, pasar inválidos a NaN y sacar features del título con expresiones regulares. Como dan siempre el mismo resultado, aplicarlos antes de separar train/test no genera fuga. Dentro del pipeline quedan solo los pasos que aprenden: el tope de superficie, el target encoding y las medianas.
2. **Modo `RAPIDO`:** una grilla chica (9 combinaciones) y validación cruzada de 3 partes, para explorar. Con `RAPIDO = False` se usa la grilla completa del notebook principal (144 combinaciones, 5 partes).

**Requisitos:** Python 3.10 o superior, `scikit-learn >= 1.4`, `pandas` y `numpy`. Los datos se leen de `datasets/` si existe esa carpeta al lado del notebook; si no, se descargan de GitHub.

In [ ]:
# ===== Configuración =====
RAPIDO = True         # True: grilla chica + 3 partes (explorar). False: grilla completa + 5 partes (resultado final)
FILTRAR_300K = True   # True: elimina del entrenamiento las casas con precio en el título > US$ 300.000 que figuran como 'no paga'
N_JOBS = -1           # núcleos a usar en la búsqueda (-1 = todos)

In [ ]:
import os, re, time, unicodedata
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, TargetEncoder
from sklearn.tree import DecisionTreeClassifier

version = tuple(int(p) for p in sklearn.__version__.split('.')[:2])
assert version >= (1, 4), f"Hace falta scikit-learn >= 1.4 (árbol con NaN y TargetEncoder). Hay {sklearn.__version__}"
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| núcleos:", os.cpu_count())

In [ ]:
URL = 'https://raw.githubusercontent.com/MartinDido100/ciencia-de-datos/refs/heads/main/primer_parcial/datasets/'
origen = 'datasets/' if os.path.isdir('datasets') else URL
df_casas_entrenamiento = pd.read_csv(origen + 'casas_entrenamiento.csv')
df_casas_predecir = pd.read_csv(origen + 'casas_predecir.csv')
print("Datos desde:", origen)
print("Entrenamiento:", df_casas_entrenamiento.shape, "| Predecir:", df_casas_predecir.shape)

## Limpieza de duplicados
Igual que en el notebook principal, y en el mismo orden:
1. Se eliminan los grupos de casas idénticas que tienen etiquetas contradictorias.
2. Se eliminan los duplicados dentro del entrenamiento.
3. Se eliminan del entrenamiento las casas idénticas a alguna de `casas_predecir`.

In [ ]:
features = [c for c in df_casas_entrenamiento.columns if c not in ('id', 'paga_comision')]
filas_inicio = len(df_casas_entrenamiento)

# 1) Casas idénticas con etiqueta contradictoria: se eliminan todas las filas del grupo
huella_tr = pd.util.hash_pandas_object(df_casas_entrenamiento[features], index=False)
contradictorias = df_casas_entrenamiento.groupby(huella_tr)['paga_comision'].transform('nunique') > 1
df_casas_entrenamiento = df_casas_entrenamiento[~contradictorias]

# 2) Duplicados de la misma casa con distinto id: queda la primera
df_casas_entrenamiento = df_casas_entrenamiento.drop_duplicates(subset=features, keep='first')

# 3) Casas de entrenamiento idénticas a alguna de predecir (predecir no se toca)
huella_tr = pd.util.hash_pandas_object(df_casas_entrenamiento[features], index=False)
huella_pr = pd.util.hash_pandas_object(df_casas_predecir[features], index=False)
df_casas_entrenamiento = df_casas_entrenamiento[~huella_tr.isin(huella_pr)]

print(f"Entrenamiento: {filas_inicio} -> {len(df_casas_entrenamiento)} filas")
assert len(df_casas_entrenamiento) == 59986, "La limpieza no da lo mismo que en el notebook principal"

## Preprocesamiento
Mismas reglas que el pipeline del notebook principal. `crear_features` se separó en dos partes:
- `features_titulo`: saca todo lo del título (precio y tipo de propiedad). No aprende nada, así que se aplica una sola vez.
- `crear_features_numericas`: va dentro del pipeline, después del tope de superficie, igual que en el original.

In [ ]:
def id_como_indice(df):
    return df.set_index('id') if 'id' in df.columns else df

entrenamiento = id_como_indice(df_casas_entrenamiento)
y = (entrenamiento['paga_comision'] == 'paga').astype(int)
X = entrenamiento.drop(columns=['paga_comision'])
X_predecir = id_como_indice(df_casas_predecir).drop(columns=['paga_comision'])

COLUMNAS_A_ELIMINAR = ['ad_type', 'l6', 'created_on', 'lat', 'lon', 'l4', 'l5',
                       'price_period', 'start_date', 'end_date']

def eliminar_columnas(X):
    return X.drop(columns=COLUMNAS_A_ELIMINAR)

def invalidos_a_nan(X):
    X = X.copy()
    X.loc[(X['bedrooms'] <= 0) | (X['bedrooms'] > 15), 'bedrooms'] = np.nan
    X.loc[X['bathrooms'] > 8, 'bathrooms'] = np.nan
    X.loc[X['surface_total'] < 20, 'surface_total'] = np.nan
    X.loc[X['surface_covered'] < 20, 'surface_covered'] = np.nan
    X.loc[X['surface_covered'] > X['surface_total'], 'surface_total'] = np.nan
    return X

PATRON_PRECIO = re.compile(r'(?:u\$s|u\$d|usd|us\$|u\$)\s*(\d[\d\.,]*)')

def normalizar_texto(texto):
    texto = unicodedata.normalize('NFKD', str(texto).lower())
    return ''.join(c for c in texto if not unicodedata.combining(c))

def extraer_precio_usd(texto):
    m = PATRON_PRECIO.search(texto)
    if not m:
        return np.nan
    numero = re.sub(r'[\.,]\d{1,2}$', '', m.group(1).rstrip('.,'))
    numero = re.sub(r'[\.,]', '', numero)
    if not numero:
        return np.nan
    valor = float(numero)
    return valor if 10_000 <= valor <= 50_000_000 else np.nan

def features_titulo(X):
    # Todo lo que sale del título. No depende de otras columnas ni aprende nada
    X = X.copy()
    titulo = X['title'].fillna('').map(normalizar_texto)
    es_alquiler = titulo.str.contains(r'\b(?:alquiler|alquilo|temporada)\b', regex=True)
    X['precio_usd'] = titulo.map(extraer_precio_usd).where(~es_alquiler)
    X['tiene_precio'] = X['precio_usd'].notna().astype(int)
    X['es_duplex'] = titulo.str.contains(r'\bduplex\b', regex=True).astype(int)
    X['es_ph'] = titulo.str.contains(r'\bph\b', regex=True).astype(int)
    X['es_depto'] = titulo.str.contains(r'\b(?:depto|dpto|departamento)s?\b', regex=True).astype(int)
    X['es_chalet'] = titulo.str.contains(r'\bchalet\b', regex=True).astype(int)
    X['tiene_piscina'] = titulo.str.contains(r'\b(?:piscina|pileta)\b', regex=True).astype(int)
    X['es_country'] = titulo.str.contains(r'\bcountry\b', regex=True).astype(int)
    X['barrio_cerrado'] = titulo.str.contains(r'\bbarrio cerrado\b', regex=True).astype(int)
    X['a_reciclar'] = titulo.str.contains(r'reciclar', regex=True).astype(int)
    return X.drop(columns=['title'])

def pasos_fijos(X):
    return features_titulo(invalidos_a_nan(eliminar_columnas(X)))

class RecortarMaximo(BaseEstimator, TransformerMixin):
    # Recorta valores absurdos al percentil indicado, calculado en fit (solo con train)
    def __init__(self, columnas=None, percentil=0.999):
        self.columnas = columnas
        self.percentil = percentil

    def fit(self, X, y=None):
        self.topes_ = {c: X[c].quantile(self.percentil) for c in self.columnas}
        self.feature_names_in_ = np.asarray(X.columns)
        return self

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_

    def transform(self, X):
        Xc = X.copy()
        for c, tope in self.topes_.items():
            Xc[c] = Xc[c].clip(upper=tope)
        return Xc

def crear_features_numericas(X):
    X = X.copy()
    X['l3'] = X['l3'].fillna('Sin dato')
    for col in ['rooms', 'bedrooms', 'bathrooms', 'surface_total', 'surface_covered']:
        X[f'{col}_faltante'] = X[col].isna().astype(int)
    X['m2_descubiertos'] = X['surface_total'] - X['surface_covered']
    rooms_orig, bedrooms_orig = X['rooms'].copy(), X['bedrooms'].copy()
    X['rooms'] = rooms_orig.fillna(bedrooms_orig + 1)
    X['bedrooms'] = bedrooms_orig.fillna((rooms_orig - 1).where(rooms_orig >= 2))
    X['surface_total'] = X['surface_total'].fillna(X['surface_covered'])
    X['surface_total'] = np.log1p(X['surface_total'])
    X['surface_covered'] = np.log1p(X['surface_covered'])
    return X

cols_numericas = ['rooms', 'bedrooms', 'bathrooms', 'surface_total', 'surface_covered', 'm2_descubiertos']
cols_binarias = ['rooms_faltante', 'bedrooms_faltante', 'bathrooms_faltante',
                 'surface_total_faltante', 'surface_covered_faltante',
                 'tiene_precio', 'es_duplex', 'es_ph', 'es_depto', 'es_chalet',
                 'tiene_piscina', 'es_country', 'barrio_cerrado', 'a_reciclar']

ct_encoding = ColumnTransformer([
    ('l1', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['l1']),
    ('l2', OneHotEncoder(min_frequency=50, handle_unknown='infrequent_if_exist', sparse_output=False), ['l2']),
    ('l3', TargetEncoder(target_type='binary', random_state=42), ['l3']),
    ('num', SimpleImputer(strategy='median'), cols_numericas),
    ('precio', 'passthrough', ['precio_usd']),
    ('bin', 'passthrough', cols_binarias),
], remainder='drop', verbose_feature_names_out=False)
ct_encoding.set_output(transform='pandas')

# Solo los pasos que aprenden de los datos (o que van después de uno que aprende)
preprocesamiento = Pipeline([
    ('recortar_cubierta', RecortarMaximo(columnas=['surface_covered'], percentil=0.999)),
    ('crear_features', FunctionTransformer(crear_features_numericas)),
    ('encoding', ct_encoding),
])

# Los pasos fijos se aplican UNA vez
inicio = time.time()
X_fijo = pasos_fijos(X)
X_predecir_fijo = pasos_fijos(X_predecir)
print(f"Pasos fijos aplicados en {time.time() - inicio:.1f} s | X: {X_fijo.shape} | X_predecir: {X_predecir_fijo.shape}")
assert list(X_fijo.columns) == list(X_predecir_fijo.columns)
assert len(X_predecir_fijo) == len(df_casas_predecir)

## Filtro: casas de más de US$ 300.000 que figuran como `no paga`
Según el enunciado, una casa paga comisión si su precio supera los US$ 300.000. Las casas de entrenamiento con precio en el título mayor a ese valor que figuran como `no paga` contradicen la regla y se eliminan. Solo se detectan las que traen el precio en el título. Se controla con `FILTRAR_300K`.

In [ ]:
if FILTRAR_300K:
    contradice_regla = (X_fijo['precio_usd'] > 300_000) & (y == 0)
    print("Casas con precio > US$ 300.000 que figuran como 'no paga':", contradice_regla.sum())
    X_modelo, y_modelo = X_fijo[~contradice_regla], y[~contradice_regla]
    assert not ((X_modelo['precio_usd'] > 300_000) & (y_modelo == 0)).any()
else:
    X_modelo, y_modelo = X_fijo, y
print(f"Filas para el modelo: {len(X_modelo)} | tasa de paga: {y_modelo.mean():.4f}")

## Árbol: búsqueda de hiperparámetros, umbral y ganancia en test
Ganancia del enunciado: +US$ 100 por cada `paga` acertado y −US$ 50 por cada falso positivo. El umbral se elige con validación cruzada sobre train; el test solo se usa para informar.

In [ ]:
def ganancia(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return 100 * ((y_pred == 1) & (y_true == 1)).sum() - 50 * ((y_pred == 1) & (y_true == 0)).sum()

assert ganancia(np.array([1, 0]), np.array([1, 1])) == 50

if RAPIDO:
    param_grid = {
        'modelo__max_depth': [10, 20, None],
        'modelo__min_samples_leaf': [5, 10, 20],
        'modelo__criterion': ['gini'],
        'modelo__class_weight': [None],
    }
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
else:
    param_grid = {
        'modelo__max_depth': [6, 8, 10, 12, 14, 16, 20, 25, None],
        'modelo__min_samples_leaf': [1, 10, 50, 100],
        'modelo__criterion': ['gini', 'entropy'],
        'modelo__class_weight': [None, 'balanced'],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X_modelo, y_modelo, test_size=0.3, stratify=y_modelo, random_state=42)

pipeline_arbol = Pipeline([
    ('preprocesamiento', clone(preprocesamiento)),
    ('modelo', DecisionTreeClassifier(random_state=42)),
])
inicio = time.time()
busqueda = GridSearchCV(pipeline_arbol, param_grid, scoring=make_scorer(ganancia), cv=cv, n_jobs=N_JOBS)
busqueda.fit(X_train, y_train)
modelo = busqueda.best_estimator_
print(f"Búsqueda: {len(busqueda.cv_results_['params'])} combinaciones x {cv.get_n_splits()} partes en {time.time() - inicio:.0f} s")
print("Mejores hiperparámetros:", busqueda.best_params_)

umbrales = np.arange(0.05, 0.95, 0.01)
proba_oof = cross_val_predict(modelo, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
umbral = float(umbrales[np.argmax([ganancia(y_train, proba_oof >= u) for u in umbrales])])

pred = modelo.predict_proba(X_test)[:, 1] >= umbral
vp = int((pred & (y_test == 1)).sum())
fp = int((pred & (y_test == 0)).sum())
fn = int((~pred & (y_test == 1)).sum())
print(f"Umbral elegido: {umbral:.2f}")
print(f"Test: {len(X_test)} casas, {int(y_test.sum())} pagan | VP: {vp} | FP: {fp} | FN: {fn}")
print(f"Ganancia del enunciado (100·VP − 50·FP): US$ {ganancia(y_test, pred):,}")
print(f"Ganancia solo de verdaderos positivos (100·VP): US$ {100 * vp:,}")
print(f"Máximo posible en test: US$ {100 * int(y_test.sum()):,}")

assert len(proba_oof) == len(X_train)   # el umbral NO se eligió con test